# MKUU AI — temporary Colab chat server

Run each cell from top to bottom. This uses the checkpoint and tokenizer already saved in Google Drive, serves the repository's own MKUU model, and creates a temporary HTTPS URL for the Android APK. Keep this Colab session open while testing.

**Privacy:** the temporary Cloudflare URL is public to anyone who has it. The notebook creates an access token; enter both the URL and token in the APK. Do not share the token.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO = Path('/content/-mkuu-foundation-model-')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-b', 'colab-t4-pilot', 'https://github.com/maxmkuu-eng/-mkuu-foundation-model-.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'colab-t4-pilot'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'colab-t4-pilot'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'colab-t4-pilot'], check=True)
print('Ready:', REPO)

In [ ]:
%cd /content/-mkuu-foundation-model-
!pip -q install -r requirements.txt -r api/requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT = Path('/content/drive/MyDrive/MKUU-Foundation-Model/checkpoints/last.pt')
TOKENIZER = Path('/content/drive/MyDrive/MKUU-Foundation-Model/mkuu_swahili_spm.model')
assert CHECKPOINT.is_file(), f'Missing checkpoint: {CHECKPOINT}'
assert TOKENIZER.is_file(), f'Missing tokenizer: {TOKENIZER}'
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Checkpoint bytes:', CHECKPOINT.stat().st_size)
print('Tokenizer bytes:', TOKENIZER.stat().st_size)

In [ ]:
import os, secrets, time, requests
os.environ['MKUU_CHECKPOINT'] = str(CHECKPOINT)
os.environ['MKUU_TOKENIZER'] = str(TOKENIZER)
os.environ['MKUU_API_TOKEN'] = secrets.token_urlsafe(32)
api_log = open('/content/mkuu-api.log', 'w')
api_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api.server:app', '--host', '0.0.0.0', '--port', '8000'], cwd=REPO, stdout=api_log, stderr=subprocess.STDOUT)
for attempt in range(90):
    try:
        health = requests.get('http://127.0.0.1:8000/health', timeout=5).json()
        if health.get('model_loaded'): break
        if health.get('load_error'): raise RuntimeError(health['load_error'])
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('API did not become ready. Check /content/mkuu-api.log')
print('Model ready:', health)

In [ ]:
import os, re, stat
cloudflared = '/content/cloudflared'
if not os.path.isfile(cloudflared):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', cloudflared], check=True)
os.chmod(cloudflared, os.stat(cloudflared).st_mode | stat.S_IEXEC)
tunnel = subprocess.Popen([cloudflared, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel URL haikupatikana. Angalia logs za tunnel.')
print('ANDROID APP API URL:', public_url)
print('ANDROID APP ACCESS TOKEN:', os.environ['MKUU_API_TOKEN'])
print('Weka URL na token hizi ndani ya APK. Usizishare na mtu mwingine.')

## Kusimamisha server
Ukimaliza majaribio, simamisha `api_process` na `tunnel` kwa kuendesha cell hii.

In [ ]:
for process in (tunnel, api_process):
    try: process.terminate()
    except Exception: pass
print('MKUU temporary chat server stopped.')